In [11]:
import pandas as pd
import numpy as np
import os
from data.preprocessing.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Pipeline
pipeline = Pipeline(feature_type="both+dft", 
                    use_pca_rdkit=False, 
                    use_pca_dft=False, 
                    remove_zero_var=True)

train_df, _ = pipeline.get_processed_data()

# Признаки и целевая переменная
X = train_df.drop(columns=['Pubchem_ID', 'taste_cluster', 'SMILES'])
y = train_df['taste_cluster']


# Разделяем на train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
Extracting raw features:   0%|          | 8/2064 [00:00<00:28, 73.22it/s][20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use MorganGenerator
[20:26:14] DEPRECATION WARNING: please use

In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros((X_train.shape[0], len(np.unique(y_train)) * 2))  # 2 модели: CatBoost + XGBoost
classes = np.unique(y_train)

cat_params = {'iterations': 500, 'learning_rate': 0.05, 'depth': 6, 'eval_metric': 'MultiClass', 'verbose': 0}
xgb_params = {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'eval_metric': 'mlogloss'}

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    
    cat = CatBoostClassifier(**cat_params, class_weights=weights)
    cat.fit(X_tr, y_tr)
    oof_preds[val_idx, :len(classes)] = cat.predict_proba(X_va)
    
    sample_weights = np.array([weights[cls] for cls in y_tr])
    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X_tr, y_tr, sample_weight=sample_weights)
    oof_preds[val_idx, len(classes):] = xgb.predict_proba(X_va)

meta_oof = np.zeros(len(y_train))
meta_params = {'iterations': 500, 'learning_rate': 0.05, 'depth': 6, 'eval_metric': 'MultiClass', 'verbose': 0}

for train_idx, val_idx in skf.split(X_train, y_train):  # Используем те же фолды
    y_tr = y_train.iloc[train_idx]
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    meta_clf = CatBoostClassifier(**meta_params, class_weights=weights)
    meta_clf.fit(oof_preds[train_idx], y_tr)
    meta_oof[val_idx] = meta_clf.predict(oof_preds[val_idx]).flatten()

f1_macro = f1_score(y_train, meta_oof, average='macro')
print(f"OOF F1 Macro: {f1_macro:.4f}")

# Если нужно финальный мета-классификатор для теста, обучаем на всех OOF
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
meta_clf = CatBoostClassifier(**meta_params, class_weights=weights)
meta_clf.fit(oof_preds, y_train)

OOF F1 Macro: 0.4192


---

In [32]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier  # Добавили LightGBM
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import BorderlineSMOTE, ADASYN  # Добавили SMOTE для опциональной аугментации

# Параметры
n_splits = 10  # Увеличили до 10 для стабильности
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Массив для out-of-fold предсказаний (теперь 3 модели: CatBoost + XGBoost + LightGBM)
classes = np.unique(y_train)
num_classes = len(classes)
oof_preds = np.zeros((X_train.shape[0], num_classes * 3))

# Базовые параметры моделей с early stopping
cat_params = {
    'iterations': 1000,  # Увеличили итерации
    'learning_rate': 0.05,
    'depth': 6,
    'eval_metric': 'TotalF1',
    'verbose': 0,
    'early_stopping_rounds': 100  # Добавили early stopping
}

xgb_params = {
    'n_estimators': 1000,  # Увеличили
    'max_depth': 6,
    'learning_rate': 0.05,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 100  # Добавили
}

lgb_params = {
    'n_estimators': 1000,  # Увеличили
    'max_depth': 6,
    'learning_rate': 0.05,
    'objective': 'multiclass',
    'num_class': num_classes,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'early_stopping_rounds': 100  # Добавили
}

In [33]:
# Опционально: Использовать SMOTE? (Включите, если imbalance сильный; иначе закомментируйте)
use_smote = True  # Поставьте False, если не нужно

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Вычисление весов классов для фолда
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    
    # Опциональная аугментация с SMOTE
    if use_smote:
        smote = BorderlineSMOTE(random_state=42, kind='borderline-1')
        X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
        # Пересчитываем веса после SMOTE (если нужно; SMOTE балансирует, так что weights можно сбросить)
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    
    sample_weights = np.array([weights[cls] for cls in y_tr])  # Веса для XGB и LGBM
    
    # CatBoost
    cat = CatBoostClassifier(**cat_params, class_weights=weights)
    cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))  # Добавили eval_set для early stopping
    
    oof_preds[val_idx, :num_classes] = cat.predict_proba(X_va)
    
    # XGBoost
    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], sample_weight=sample_weights, verbose=False)
    oof_preds[val_idx, num_classes:num_classes*2] = xgb.predict_proba(X_va)
    
    # LightGBM (добавили)
    lgb = LGBMClassifier(**lgb_params)
    lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], sample_weight=sample_weights)
    oof_preds[val_idx, num_classes*2:] = lgb.predict_proba(X_va)


In [ ]:

# Альтернатива 1: Soft Voting (усреднение вероятностей) вместо мета-классификатора
# avg_probs = (oof_preds[:, :num_classes] + oof_preds[:, num_classes:num_classes*2] + oof_preds[:, num_classes*2:]) / 3
# final_preds_vote = np.argmax(avg_probs, axis=1)
# f1_macro_vote = f1_score(y_train, final_preds_vote, average='macro')
# print(f"OOF F1 Macro (Voting): {f1_macro_vote:.4f}")

# Альтернатива 2: Blending (разделение train на train1/train2)
X_tr1, X_tr2, y_tr1, y_tr2 = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

# Для blending: Используем oof_preds как фичи, но на X_tr2
# (Предполагаем, что oof_preds уже собраны; для чистоты blending можно переобучить базы на X_tr1 и предсказать на X_tr2, но для простоты используем full oof)

In [35]:
from sklearn.model_selection import GridSearchCV  # Или import optuna

# Пример для CatBoost (добавьте перед циклом)
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5]
}
grid_search = GridSearchCV(CatBoostClassifier(**cat_params, class_weights=weights), param_grid, scoring='f1_macro', cv=3)
grid_search.fit(X_tr, y_tr)  # На одном фолде для теста
best_params = grid_search.best_params_
print("Best params:", best_params)
# Затем обновите cat_params = {**cat_params, **best_params}

KeyboardInterrupt: 

In [ ]:

# Теперь строим мета-классификатор с кросс-валидацией для правильной OOF оценки
meta_oof = np.zeros(len(y_train))
meta_params = {
    'iterations': 1000,  # Увеличили
    'learning_rate': 0.05,
    'depth': 6,
    'eval_metric': 'TotalF1',
    'verbose': 0,
    'early_stopping_rounds': 100  # Добавили
}

In [ ]:
for train_idx, val_idx in skf.split(X_train, y_train):  # Используем те же фолды
    X_meta_tr, X_meta_va = oof_preds[train_idx], oof_preds[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    
    meta_clf = CatBoostClassifier(**meta_params, class_weights=weights)
    meta_clf.fit(X_meta_tr, y_tr, eval_set=(X_meta_va, y_va))  # Добавили eval_set
    
    meta_oof[val_idx] = meta_clf.predict(X_meta_va).flatten()

# F1 Macro на правильной OOF
f1_macro = f1_score(y_train, meta_oof, average='macro')
print(f"OOF F1 Macro (Stacking): {f1_macro:.4f}")

# Для теста: Сначала получите preds от баз на test, затем meta.predict(test_preds)
# (Код для test не включен, добавьте по аналогии)

In [ ]:
# Если нужно финальный мета-классификатор для теста, обучаем на всех OOF
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
meta_clf = CatBoostClassifier(**meta_params, class_weights=weights)
meta_clf.fit(oof_preds, y_train)